`Data generation workbook`

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# --- Configuration ---
N_RAW_SAMPLES = 1500  # Number of samples for raw_energy_data and model_data
N_VAL_SAMPLES = 500   # Number of samples for validation_features
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# --- Helper Functions ---
def generate_base_data(num_samples):
    """Generates a base DataFrame with clean data."""
    data = pd.DataFrame()
    data['household_id'] = range(1, num_samples + 1)
    data['time_of_use'] = np.random.uniform(1, 24, num_samples).round(2)
    data['appliances_used'] = np.random.randint(1, 15, num_samples)
    data['energy_cost'] = np.random.uniform(0.5, 10.0, num_samples).round(2)
    data['home_type'] = np.random.choice(['Apartment', 'House', 'Condo'], num_samples, p=[0.4, 0.4, 0.2])
    data['occupancy_type'] = np.random.choice(['Single', 'Family'], num_samples, p=[0.45, 0.55])

    # Make energy_usage dependent on features
    score = (
        0.25 * (data['time_of_use'] / 24) +
        0.20 * (data['appliances_used'] / 15) +
        0.20 * (data['energy_cost'] / 10) +
        data['home_type'].apply(lambda x: 0.15 if x == 'House' else (-0.05 if x == 'Apartment' else 0)) +
        data['occupancy_type'].apply(lambda x: 0.20 if x == 'Family' else 0)
    )
    # Introduce some noise and ensure the score is not too skewed
    score_mean = score.mean()
    score_std = score.std()
    normalized_score = (score - score_mean) / (score_std if score_std > 0 else 1) # Normalize score
    normalized_score += np.random.normal(0, 0.2, num_samples) # Add some noise

    probability = 1 / (1 + np.exp(-normalized_score)) # Sigmoid
    data['energy_usage'] = (probability > 0.55).astype(int) # Adjust threshold for balance

    return data

def introduce_imperfections(df_clean):
    """Introduces imperfections to a clean DataFrame to create raw_data."""
    df_raw = df_clean.copy()

    # 1. `time_of_use`: Missing values (float, np.nan)
    missing_time_indices = np.random.choice(df_raw.index, size=int(N_RAW_SAMPLES * 0.1), replace=False)
    df_raw.loc[missing_time_indices, 'time_of_use'] = np.nan

    # 2. `appliances_used`: Type conversion (int to string) and some missing values
    missing_appliances_indices = np.random.choice(df_raw.index, size=int(N_RAW_SAMPLES * 0.08), replace=False)
    df_raw.loc[missing_appliances_indices, 'appliances_used'] = np.nan
    df_raw['appliances_used'] = df_raw['appliances_used'].astype(str)

    # 3. `energy_cost`: Missing values (float, np.nan)
    missing_cost_indices = np.random.choice(df_raw.index, size=int(N_RAW_SAMPLES * 0.07), replace=False)
    df_raw.loc[missing_cost_indices, 'energy_cost'] = np.nan

    # 4. `home_type`: Text mistake and missing values (string, "N/A")
    mistake_indices = df_raw[df_raw['home_type'] == 'Apartment'].sample(frac=0.1, random_state=RANDOM_SEED).index
    df_raw.loc[mistake_indices, 'home_type'] = 'Apartement' # Text mistake
    missing_home_type_indices = np.random.choice(df_raw.index, size=int(N_RAW_SAMPLES * 0.09), replace=False)
    missing_home_type_indices = np.setdiff1d(missing_home_type_indices, mistake_indices)
    df_raw.loc[missing_home_type_indices, 'home_type'] = "N/A" # String missing value

    # 5. `occupancy_type`: Missing values (np.nan, to be filled with "Single")
    missing_occupancy_indices = np.random.choice(df_raw.index, size=int(N_RAW_SAMPLES * 0.06), replace=False)
    df_raw.loc[missing_occupancy_indices, 'occupancy_type'] = np.nan # Will be actual NaN

    return df_raw

def clean_data_task1(df_raw):
    """Applies cleaning rules as per Task 1 to generate model_data."""
    df_cleaned = df_raw.copy()

    median_time_of_use = df_cleaned['time_of_use'].median()
    df_cleaned['time_of_use'].fillna(median_time_of_use, inplace=True)

    df_cleaned['appliances_used'] = pd.to_numeric(df_cleaned['appliances_used'], errors='coerce')
    mean_appliances_used = df_cleaned['appliances_used'].mean()
    df_cleaned['appliances_used'].fillna(mean_appliances_used, inplace=True)
    df_cleaned['appliances_used'] = df_cleaned['appliances_used'].astype(int)

    df_cleaned['energy_cost'].fillna(0, inplace=True)

    df_cleaned['home_type'] = df_cleaned['home_type'].replace('Apartement', 'Apartment')
    df_cleaned['home_type'] = df_cleaned['home_type'].replace('N/A', 'Unknown')
    df_cleaned['home_type'].fillna("Unknown", inplace=True)

    df_cleaned['occupancy_type'].fillna("Single", inplace=True)

    return df_cleaned

def prepare_features_task2(df, scaler=None, train_cols=None):
    """Applies feature engineering as per Task 2."""
    df_featured = df.copy()

    numerical_cols = ['time_of_use', 'appliances_used', 'energy_cost']
    categorical_cols = ['home_type', 'occupancy_type']

    if scaler is None:
        scaler = MinMaxScaler()
        df_featured[numerical_cols] = scaler.fit_transform(df_featured[numerical_cols])
    else:
        df_featured[numerical_cols] = scaler.transform(df_featured[numerical_cols])

    df_featured = pd.get_dummies(df_featured, columns=categorical_cols, prefix=categorical_cols, prefix_sep='_')

    if train_cols is not None:
        missing_cols = set(train_cols) - set(df_featured.columns)
        for c in missing_cols:
            if c not in ['household_id', 'energy_usage']:
                 df_featured[c] = 0
        df_featured = df_featured[train_cols]

    return df_featured, scaler

# --- Main Script ---

base_data_for_raw = generate_base_data(N_RAW_SAMPLES)

raw_energy_data = introduce_imperfections(base_data_for_raw)
raw_energy_data.to_csv('raw_energy_data.csv', index=False)

model_data = clean_data_task1(raw_energy_data.copy())
cols_order = ['household_id', 'time_of_use', 'appliances_used', 'energy_cost', 'home_type', 'occupancy_type', 'energy_usage']
model_data = model_data[cols_order]
model_data.to_csv('model_data.csv', index=False)

input_model_features, scaler_for_validation = prepare_features_task2(model_data.copy())
input_model_features.to_csv('input_model_features.csv', index=False)
train_feature_columns = input_model_features.columns.tolist()

base_validation_data = generate_base_data(N_VAL_SAMPLES)
base_validation_data['household_id'] = base_validation_data['household_id'] + N_RAW_SAMPLES

validation_features, _ = prepare_features_task2(base_validation_data.copy(), scaler=scaler_for_validation, train_cols=train_feature_columns)
validation_subset = validation_features[['household_id', 'energy_usage']].values
validation_features.drop(columns='energy_usage').to_csv('validation_features.csv', index=False)


In [3]:
# Format the NumPy array as a string to match the requested output style
# print("np.array([", end="")
# for i, row in enumerate(validation_subset):
#     if i > 0:
#         print(",", end="")
#     print(f"[{int(row[0])},{int(row[1])}]", end="")
# print("])")
